<a href="https://colab.research.google.com/github/GiuseppeMinissale/Second-Project-Real-vs-AI-Art-Classification/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
from google.colab import userdata

# Carica il nuovo token singolo in modo sicuro
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

# Scarica e decomprimi il dataset
!kaggle datasets download -d ravidussilva/real-ai-art
!unzip -q real-ai-art.zip

Dataset URL: https://www.kaggle.com/datasets/ravidussilva/real-ai-art
License(s): other
100% 9.95G/9.95G [01:38<00:00, 109MB/s]



In [2]:
!ls -l

total 10431588
-rw-r--r-- 1 root root 10681931755 Aug 16  2023 real-ai-art.zip
drwxr-xr-x 4 root root        4096 Jun 22 13:10 Real_AI_SD_LD_Dataset
drwxr-xr-x 1 root root        4096 Jun  4 13:39 sample_data


In [3]:
import os
import json
import random
import hashlib
from pathlib import Path
from PIL import Image, ImageOps
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
import torchvision

# ==========================================
# BLOCCO 1: Impostazioni Iniziali
# ==========================================
DATA_DIR = Path("Real_AI_SD_LD_Dataset")
CACHE_FILE = Path("dataset_cache_split.json") # Nome aggiornato per la nuova struttura

if not DATA_DIR.exists():
    possible_dirs = [d for d in Path(".").iterdir() if d.is_dir() and "Real_AI" in d.name]
    if possible_dirs: DATA_DIR = possible_dirs[0]

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"

# ==========================================
# BLOCCO 2: Gli Strumenti di Pulizia
# ==========================================
def calcola_hash_md5(percorso_file, chunk_size=8192):
    hash_md5 = hashlib.md5()
    try:
        with open(percorso_file, "rb") as f:
            for chunk in iter(lambda: f.read(chunk_size), b""):
                hash_md5.update(chunk)
        return hash_md5.hexdigest()
    except Exception: return None

def esegui_data_cleaning_su_lista(lista_percorsi, max_aspect_ratio_deviation=3.0, min_std_deviation=0.01):
    percorsi_puliti = []
    hash_visti = set()
    for file_path in lista_percorsi:
        try:
            with Image.open(file_path) as img: img.verify()
            with Image.open(file_path) as img:
                file_hash = calcola_hash_md5(file_path)
                if file_hash is None or file_hash in hash_visti: continue
                img = ImageOps.exif_transpose(img)
                w, h = img.size
                if w < 64 or h < 64: continue
                aspect_ratio = float(h) / float(w)
                if aspect_ratio > max_aspect_ratio_deviation or aspect_ratio < (1.0 / max_aspect_ratio_deviation): continue
                img_np = np.array(img.convert("RGB")) / 255.0
                if np.std(img_np) < min_std_deviation: continue

                hash_visti.add(file_hash)
                percorsi_puliti.append(str(file_path))
        except Exception: continue
    return percorsi_puliti

# Funzione di supporto per scansionare una specifica cartella
def scansiona_e_classifica(directory):
    c_R, c_A = [], []
    estensioni = ['.jpg', '.jpeg', '.png']
    tutti_i_file = []
    for ext in estensioni:
        tutti_i_file.extend(directory.rglob(f"*{ext}"))
        tutti_i_file.extend(directory.rglob(f"*{ext.upper()}"))

    for p in tutti_i_file:
        if any(k in str(p).lower() for k in ['real_ai_sd_ld_dataset/test/ai', 'real_ai_sd_ld_dataset/train/ai']):
            c_A.append(p)
        else:
            c_R.append(p)
    return c_R, c_A

# ==========================================
# BLOCCO 3: Il Bivio Logico (Isolamento Train/Test)
# ==========================================
if CACHE_FILE.exists():
    print(f"Trovato file {CACHE_FILE}. Caricamento in corso...")
    with open(CACHE_FILE, 'r') as f:
        cache_data = json.load(f)
        train_paths, train_labels = cache_data['train_paths'], cache_data['train_labels']
        test_paths, test_labels = cache_data['test_paths'], cache_data['test_labels']
else:
    print("Nessun salvataggio trovato. Avvio estrazione differenziata...")
    if not TRAIN_DIR.exists() or not TEST_DIR.exists():
        raise FileNotFoundError("ERRORE: Cartelle 'train' o 'test' mancanti.")

    # 1. Processiamo il Train Set
    print("Analisi cartella TRAIN...")
    tr_cand_R, tr_cand_A = scansiona_e_classifica(TRAIN_DIR)
    tr_R_puliti = esegui_data_cleaning_su_lista(tr_cand_R)
    tr_A_puliti = esegui_data_cleaning_su_lista(tr_cand_A)

    K_train = min(2400, len(tr_R_puliti), len(tr_A_puliti))
    random.seed(42)
    train_paths = random.sample(tr_R_puliti, K_train) + random.sample(tr_A_puliti, K_train)
    train_labels = [0] * K_train + [1] * K_train

    # 2. Processiamo il Test Set
    print("Analisi cartella TEST...")
    te_cand_R, te_cand_A = scansiona_e_classifica(TEST_DIR)
    te_R_puliti = esegui_data_cleaning_su_lista(te_cand_R)
    te_A_puliti = esegui_data_cleaning_su_lista(te_cand_A)

    K_test = min(600, len(te_R_puliti), len(te_A_puliti))
    test_paths = random.sample(te_R_puliti, K_test) + random.sample(te_A_puliti, K_test)
    test_labels = [0] * K_test + [1] * K_test

    # Salvataggio separato
    with open(CACHE_FILE, 'w') as f:
        json.dump({
            'train_paths': train_paths, 'train_labels': train_labels,
            'test_paths': test_paths, 'test_labels': test_labels
        }, f)
    print(f"Pulizia terminata. Cache salvata in {CACHE_FILE}.")

# ==========================================
# BLOCCO 4: Preparazione per la Rete Neurale (Senza random_split)
# ==========================================
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class ArtDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths, self.labels, self.transform = paths, labels, transform
    def __len__(self): return len(self.paths)
    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(image), torch.tensor(self.labels[idx], dtype=torch.float32)

# Inizializziamo i Dataset separatamente utilizzando i percorsi divisi
train_dataset = ArtDataset(train_paths, train_labels, transform)
val_dataset = ArtDataset(test_paths, test_labels, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Dataset allocato fedelmente: {len(train_dataset)} training (Da cartella 'train'), {len(val_dataset)} validation (Da cartella 'test').")

Nessun salvataggio trovato. Avvio estrazione differenziata...
Analisi cartella TRAIN...
Analisi cartella TEST...
Pulizia terminata. Cache salvata in dataset_cache_split.json.
Dataset allocato fedelmente: 4800 training (Da cartella 'train'), 1200 validation (Da cartella 'test').
